# **LangGraph Retrieval Agent**

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs_list)

# Add to vectorDB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()

In [ ]:
from langchain_core.tools import create_retriever_tool
tool = create_retriever_tool(
    retriever,
    "retrieve_blog_posts",
    "Search and return information about Lilian Weng blog posts on LLM agents, prompt engineering, and adversarial attacks on LLMs.",
)
tools = [tool]
from langgraph.prebuilt import ToolNode
tool_node = ToolNode(tools)

In [ ]:
import operator
from typing import Annotated,Sequence,TypedDict
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages : Annotated[Sequence[BaseMessage] , operator.add]

In [3]:
import json 
import pprint
from typing import Annotated, Sequence, TypedDict
from pydantic import BaseModel, Field

from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph, START

In [4]:
class AgentState(TypedDict):
    # Using the standard messages list reducer
    messages: Annotated[Sequence[BaseMessage], lambda x, y: x + list(y)]

# Dummy retriever tool for execution demonstration
def retriever_tool(query: str) -> str:
    return f"Mocked documents matching: '{query}'. Lilian Weng discusses Short-term memory (in-context learning) and Long-term memory (external vector stores)."

tools = [retriever_tool]

In [6]:
def should_retrieve(state: AgentState):
    """
    Decides whether the agent should retrieve more information or end the process.
    Checks for tool calls in the modern AIMessage format.
    """
    print("---DECIDE TO RETRIEVE---")
    messages = state["messages"]
    last_message = messages[-1]
    
    # Check if the model requested any tool calls
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        print("---DECISION: RETRIEVE---")
        return "continue"
    else:
        print("---DECISION: DO NOT RETRIEVE / DONE---")
        return "end"


def grade_documents(state: AgentState):
    """
    Determines whether the retrieved documents are relevant to the question.
    """
    print("---CHECK RELEVANCE---")

    # Data model - Modern Pydantic v2
    class Grade(BaseModel):
        """Binary score for relevance check."""
        binary_score: str = Field(description="Relevance score 'yes' or 'no'")

    # Modern structured output method using .with_structured_output()
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini")
    structured_llm = model.with_structured_output(Grade)

    prompt = PromptTemplate(
        template="""You are a grader assessing relevance of a retrieved document to a user question.
Here is the retrieved document:

{context}

Here is the user question: {question}
If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant.
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.""",
        input_variables=["context", "question"],
    )

    chain = prompt | structured_llm

    messages = state["messages"]
    question = messages[0].content
    
    # Find the last tool message containing documents
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    docs = tool_messages[-1].content if tool_messages else ""
    
    score = chain.invoke({"question": question, "context": docs})
    grade = score.binary_score.lower().strip()

    if grade == "yes":
        print("---DECISION: DOCS RELEVANT---")
        return "yes"
    else:
        print("---DECISION: DOCS NOT RELEVANT---")
        return "no"


# ==========================================
# II. NODES
# ==========================================

def agent(state: AgentState):
    """
    Invokes the agent model to decide to retrieve or respond using bind_tools.
    """
    print("---CALL AGENT---")
    messages = state["messages"]
    
    # Modern approach: use .bind_tools() instead of legacy functions
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini").bind_tools(tools)
    response = model.invoke(messages)
    return {"messages": [response]}


def retrieve(state: AgentState):
    """
    Executes the modern Tool Call handling.
    """
    print("---EXECUTE RETRIEVAL---")
    messages = state["messages"]
    last_message = messages[-1]
    
    tool_messages = []
    # Execute tools requested by the model
    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "retriever_tool":
            # Extract arguments cleanly
            query = tool_call["args"].get("query", "")
            response_content = retriever_tool(query)
            
            # Construct standard ToolMessage returning the output back to graph
            tool_messages.append(
                ToolMessage(
                    content=str(response_content), 
                    tool_call_id=tool_call["id"],
                    name=tool_call["name"]
                )
            )
            
    return {"messages": tool_messages}


def rewrite(state: AgentState):
    """
    Transform the query to produce a better question.
    """
    print("---TRANSFORM QUERY---")
    messages = state["messages"]
    question = messages[0].content

    msg = [HumanMessage(
        content=f"Look at the input and try to reason about the underlying semantic intent / meaning.\nHere is the initial question:\n-------\n{question}\n-------\nFormulate an improved question:"
    )]

    model = ChatOpenAI(temperature=0, model="gpt-4o-mini")
    response = model.invoke(msg)
    return {"messages": [response]}


def generate(state: AgentState):
    """
    Generate the final answer.
    """
    
    messages = state["messages"]
    question = messages[0].content

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    docs = tool_messages[-1].content if tool_messages else ""

    # Your own RAG prompt
    prompt = PromptTemplate.from_template(
        """
            You are a helpful assistant. Answer the user's question using only the provided context.

            Context:
            {context}

            Question:
            {question}

            Instructions:
            - If the answer is not in the context, say "I don't know based on the provided information."
            - Keep the answer concise and accurate.
            - Do not make up information.

            Answer:
            """
    )

    llm = ChatOpenAI(
        model_name="gpt-4o-mini",
        temperature=0
    )

    rag_chain = prompt | llm | StrOutputParser()

    response = rag_chain.invoke(
        {
            "context": docs,
            "question": question
        }
    )

    return {
        "messages": [
            AIMessage(content=response)
        ]
    }


# ==========================================
# III. GRAPH COMPILATION
# ==========================================

workflow = StateGraph(AgentState)

# Define nodes
workflow.add_node("agent", agent)
workflow.add_node("retrieve", retrieve)
workflow.add_node("rewrite", rewrite)
workflow.add_node("generate", generate)

# Flow Setup
workflow.add_edge(START, "agent")

workflow.add_conditional_edges(
    "agent",
    should_retrieve,
    {
        "continue": "retrieve",
        "end": END,
    },
)

workflow.add_conditional_edges(
    "retrieve",
    grade_documents,
    {
        "yes": "generate",
        "no": "rewrite",  
    },
)

workflow.add_edge("generate", END)
workflow.add_edge("rewrite", "agent")

# Compile Graph
app = workflow.compile()

# ==========================================
# IV. EXECUTION
# ==========================================

inputs = {
    "messages": [
        HumanMessage(
            content="What does Lilian Weng say about the types of agent memory?"
        )
    ]
}

for output in app.stream(inputs):
    for key, value in output.items():
        pprint.pprint(f"Output from node '{key}':")
        pprint.pprint("---")
        pprint.pprint(value, indent=2, width=80, depth=None)
    pprint.pprint("\n---\n")

---CALL AGENT---
---DECIDE TO RETRIEVE---
---DECISION: RETRIEVE---
"Output from node 'agent':"
'---'
{ 'messages': [ AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 52, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_303afd94e3', 'id': 'chatcmpl-DvmbT7AxqvmnULVXCbVpHZ4rynVyw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f0f0c-6544-7fa3-be90-8ad031fed713-0', tool_calls=[{'name': 'retriever_tool', 'args': {'query': 'Lilian Weng types of agent memory'}, 'id': 'call_Oh3rZBNGGCaPJo86KVDtwbBc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens':